# — CLIP + Qwen3 image captioning

The happy path end to end, in the order the README describes it. Every cell calls the
installed package rather than redefining the pipeline: the training loop below is the
same `train()` that `make train` runs, and the captions come from the same
`generate_caption()` the Streamlit UI and the FastAPI service call.

That matters here specifically. This repo used to keep one copy of the training loop in
this notebook and a second, different one in `final.ipynb`, and the two silently drifted
apart — which is how the checkpoint/adapter mismatch documented in README Section 6 got
in. There is one copy now.

In [1]:
import tempfile
from pathlib import Path

from PIL import Image

from captioning import generate_caption
from config import CHECKPOINTS, Config
from models import build_adapter, load_backbones
from training import train
from zero_shot import classify_images

# Jupyter sets the kernel's working directory to this notebook's own folder, so the
# paths are relative to notebooks/ rather than the repo root.
config = Config(checkpoint_dir=Path("../checkpoints"), data_dir=Path("../data"))
config

Config(seed=42, batch_size=8, learning_rate=0.0001, num_epochs=3, checkpoint_dir=PosixPath('../checkpoints'), data_dir=PosixPath('../data'), hf_repo_id='RoshBeed/ai-residency-multimodal-captioning', wandb_project='ai-residency-multimodal-captioning', clip_model_name='openai/clip-vit-base-patch32', qwen_model_name='Qwen/Qwen3-0.6B-Base', dataset_name='nlphuji/flickr30k', dataset_revision='refs/convert/parquet', dataset_split='test', num_samples=100, train_fraction=0.8, max_caption_length=32, final_layer_norm=False, checkpoint_name='adapter_flickr_batched.pth', train_on_cpu=True, prompt='This is a photo of:', max_new_tokens=30, zero_shot_items=('cat', 'dog', 'horse', 'bird', 'car', 'person', 'tree', 'house', 'book', 'phone'), api_host='0.0.0.0', api_port=8000)

## Part 1 — zero-shot classification with CLIP alone

No training and no adapter. Each label becomes the sentence "a photo of a {label}", CLIP
embeds it in the same space as the image, and the highest dot product wins. Adding a new
class costs one sentence rather than a labelled dataset, which is the whole argument for
a joint image/text embedding space.

In [2]:
for prediction in classify_images(config):
    mark = "ok " if prediction.correct else "MISS"
    print(f"{mark} {prediction.item:8s} -> {prediction.predicted:8s} ({prediction.score:.3f})")

14:47:31 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


14:47:31 INFO     httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"


14:47:31 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"


14:47:31 WARNING  huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


14:47:31 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"


...

14:47:34 INFO     httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-0.6B-Base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


14:47:34 INFO     httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-0.6B-Base/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


14:47:35 INFO     httpx | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-0.6B-Base "HTTP/1.1 200 OK"


14:47:35 INFO     models | Loaded openai/clip-vit-base-patch32 (512-d) and Qwen/Qwen3-0.6B-Base (1024-d) on mps


ok  cat      -> cat      (30.386)
ok  dog      -> dog      (36.011)
ok  horse    -> horse    (36.965)
ok  bird     -> bird     (30.406)
ok  car      -> car      (23.988)
ok  person   -> person   (27.898)
ok  tree     -> tree     (30.657)
ok  house    -> house    (29.411)
ok  book     -> book     (40.089)
ok  phone    -> phone    (32.120)


## Part 2 — the adapter

The only trainable component. CLIP hands over a 512-d image embedding; Qwen3 reads 1024-d
token embeddings. The adapter is the projection between them, and its output is fed to
Qwen3 as a single soft token, as if the decoder had read one word describing the picture.

`final_layer_norm` is the flag that fixes this repo's longest-standing bug.
`adapter_flickr_batched.pth` was trained with a trailing `LayerNorm` that was later
deleted from the code, and nothing recorded that, so loading it raised
`Unexpected key(s) in state_dict: "4.weight", "4.bias"`. Each checkpoint now carries its
own architecture.

In [3]:
backbones = load_backbones(config)
adapter = build_adapter(backbones.clip_dim, backbones.qwen_dim)

print(f"CLIP embedding width: {backbones.clip_dim}")
print(f"Qwen3 embedding width: {backbones.qwen_dim}")
print(f"Trainable adapter parameters: {sum(p.numel() for p in adapter.parameters()):,}")
frozen = sum(p.numel() for p in backbones.qwen_model.parameters())
print(f"Frozen Qwen3 parameters:      {frozen:,}")

for name, info in CHECKPOINTS.items():
    print(f"{name:30s} final_layer_norm={info.final_layer_norm}  {info.description}")

CLIP embedding width: 512
Qwen3 embedding width: 1024
Trainable adapter parameters: 1,576,960
Frozen Qwen3 parameters:      596,049,920
adapter_flickr_batched.pth     final_layer_norm=True  Flickr30k, 100 pairs / 3 epochs, batched (best-trained)
adapter.pth                    final_layer_norm=False  Flickr30k, 10 pairs / 3 epochs, unbatched at 1e-5 (barely moved off init)
adapter_trained.pth            final_layer_norm=False  10 local images / 20 epochs (overfit — degenerates into repeated tokens)


## Training the adapter

The real batched loop: a `DataLoader` over Flickr30k pairs with a `collate_fn` that
processes whole batches of images and pads the captions together. Both backbones stay
frozen, so gradients only ever reach the adapter.

The baseline is the same held-out loss measured with an untrained adapter. It is worth
computing because Qwen3 is fluent whatever the image token contains, so a caption loss
with nothing to compare it against says very little.

Training pins CPU on purpose — see `Config.train_on_cpu` for the measurement behind that.
This cell writes to a scratch directory so it cannot overwrite the committed checkpoints.

In [4]:
scratch = Path(tempfile.gettempdir()) / "ai-residency-multimodal-captioning-walkthrough"
best_loss = train(
    Config(
        checkpoint_dir=scratch,
        data_dir=Path("../data"),
        num_samples=40,
        num_epochs=2,
        wandb_project=None,
        hf_repo_id=None,
    )
)
print(f"Best held-out loss: {best_loss:.4f}")

14:47:35 INFO     ai_residency_common.config | Seeded RNGs with 42


14:47:35 INFO     training | Training on cpu


14:47:36 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


14:47:36 INFO     httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"


14:47:36 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"


...

epoch 2/2: 100%|██████████| 4/4 [00:04<00:00,  1.18s/it]

14:47:52 INFO     training | Epoch 2/2  train_loss=5.0724  val_loss=5.8407


14:47:52 INFO     ai_residency_common.checkpoints | Saved checkpoint to /var/folders/rz/2164hm0x2cv9dcqz41x_4l640000gn/T/ai-residency-multimodal-captioning-walkthrough/adapter_flickr_batched.pth


14:47:52 INFO     training | Best held-out loss 5.8407 against baseline 7.4877 (beats baseline)


Best held-out loss: 5.8407


## Captioning, across all three committed checkpoints

The honest result. Every caption below is real greedy-decoded output from the actual
CLIP → adapter → Qwen3 path, and none of them describe the image. ~1.6M adapter
parameters trained on 100 pairs cannot override a 596M-parameter decoder's own priors, so
Qwen3 stays fluent and stops looking at the picture. Fluent output from a model that is
barely conditioned on its input is the thing worth seeing here.

In [5]:
for checkpoint_name in CHECKPOINTS:
    print(f"\n=== {checkpoint_name} ===")
    for item in ("cat", "dog", "book"):
        image = Image.open(config.image_dir / f"{item}.jpg")
        caption = generate_caption(image, config, checkpoint_name=checkpoint_name)
        print(f"  {item}.jpg -> {caption!r}")


=== adapter_flickr_batched.pth ===
14:47:52 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


14:47:52 INFO     httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/openai/clip-vit-base-patch32/3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268/config.json "HTTP/1.1 200 OK"


14:47:52 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors "HTTP/1.1 404 Not Found"


14:47:52 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/model.safetensors.index.json "HTTP/1.1 404 Not Found"


14:47:52 INFO     httpx | HTTP Request: HEAD https://huggingface.co/openai/clip-vit-base-patch32/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"


...

  dog.jpg -> '1999-2000 season. The 1999-2000 season was the 10th'
14:48:00 INFO     ai_residency_common.checkpoints | Loaded checkpoint from ../checkpoints/adapter.pth


  book.jpg -> '1999-2000 season. The team was the 1999-2000 season was the'

=== adapter_trained.pth ===
14:48:01 INFO     ai_residency_common.checkpoints | Loaded checkpoint from ../checkpoints/adapter_trained.pth


  cat.jpg -> ']:::::::::::::::::::::::::::::::::::::::::::::::::::::::::::'
14:48:02 INFO     ai_residency_common.checkpoints | Loaded checkpoint from ../checkpoints/adapter_trained.pth


  dog.jpg -> ':::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::'
14:48:03 INFO     ai_residency_common.checkpoints | Loaded checkpoint from ../checkpoints/adapter_trained.pth


  book.jpg -> ':\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:\n:'
